In [ ]:
%%html
<!-- This cell defines CSS styles for the notebook. -->
<style>
    /* Make sure tab titles are the length of their text. */
    .lm-TabBar-tab {flex-basis: auto !important;}
</style>

In [ ]:
#import sys
#! {sys.executable} -m pip install duckdb
#! {sys.executable} -m pip install --force-reinstall --no-deps git+https://github.com/aaronweeden/xdmod-data.git@whoami

In [ ]:
from datetime import datetime
import duckdb
from IPython.display import display, Markdown
import ipywidgets
import pandas as pd
import plotly.express as px
import plotly.io as pio
from xdmod_data.warehouse import DataWarehouse
import xdmod_data.themes
pio.renderers.default = 'plotly_mimetype+notebook'
pio.templates.default = 'timeseries'

In [ ]:
dw = DataWarehouse('https://xdmod-dev.ccr.xdmod.org:9001')
with dw:
    whoami = dw.whoami()
    # TODO: These could be moved to the same script that generates the parquet file.
    all_projects = dw.get_filter_values('Jobs', 'Allocation')
    all_resources = dw.get_filter_values('Jobs', 'Resource')
all_projects.index = all_projects.index.astype(int)
all_resources.index = all_resources.index.astype(int)
#person_id = whoami['person_id']
person_id = 294434
def get_data(select='*', where=None, group_by=None):
    query = f"PRAGMA disable_optimizer; SELECT {select} FROM '/shared/test.parquet'"
    if where is not None:
        query += ' WHERE ' + where
    if group_by is not None:
        query += ' GROUP BY ' + group_by
    return duckdb.query(query).to_df()
my_df = get_data(where=f'person_id = {person_id} or pi_id = {person_id}')
my_project_ids = my_df['allocation_id'].unique()
my_project_charge_numbers_and_titles = all_projects.loc[my_project_ids, 'label'].str.split(' - ').to_dict()
#print(my_project_ids)
#print(my_project_charge_numbers_and_titles)
#print(my_df)

In [ ]:
Tabs = {
    'projects': {
        'tab_widget': ipywidgets.Tab(),
    },
}

def build_projects_tab_widget():
    projects_tab_widget = Tabs['projects']['tab_widget']
    projects_tab_children = []
    projects_tab_titles = []
    for project_id, charge_number_and_title in my_project_charge_numbers_and_titles.items():
        if len(charge_number_and_title) == 1:
            charge_number = title = charge_number_and_title[0]
        else:
            (charge_number, title) = charge_number_and_title
        output = ipywidgets.Output()
        projects_tab_children.append(output)
        projects_tab_titles.append(charge_number)
        project_tab_widget = ipywidgets.Tab()
        project_tab_widget_titles = [
            'My Usage',
            'Total Project Usage',
            'Project Usage By User',
        ]
        project_tab_widget.children = [
            build_tab_widget(
                ['Total', 'By Resource'],
                [
                    build_timeseries_plot_with_total(f'person_id = {person_id} and allocation_id = {project_id}'),
                    build_tab_widget(
                        ['Total', 'By Day'],
                        build_aggregate_and_timeseries_tab(
                            where=f'person_id = {person_id} and allocation_id = {project_id}',
                            group_by='resource_id',
                        ),
                    ),
                ],
            ),
            build_tab_widget(
                ['Total', 'By User'],
                [
                    build_timeseries_plot_with_total(f'allocation_id = {project_id}'),
                ],
            ),
        ]
        project_tab_widget.titles = project_tab_widget_titles
        Tabs['projects']['sub_tabs'] = {
            'charge_number': {
                'tab_widget': project_tab_widget,
            },
        }
        with output:
            display(Markdown(f'### {title}'))
            display(project_tab_widget)
    projects_tab_widget.children = projects_tab_children
    projects_tab_widget.titles = projects_tab_titles
    return projects_tab_widget

def build_timeseries_plot_with_total(where):
    output = ipywidgets.Output()
    df = get_data(where=where)
    df['Day'] = df['day_id'].apply(lambda x: datetime.strptime(str(x), "%Y00%j").day)
    start = pd.Timestamp('2025-04-01')
    end = start + pd.offsets.MonthEnd(0)
    df = (
        df.rename(columns={'total_ace': 'ACCESS Credit Equivalents'})
          .groupby('Day', as_index=False)['ACCESS Credit Equivalents'].sum()
          .set_index('Day')
          .reindex(range(1, end.day + 1), fill_value=0)
          .rename_axis('Day')
    )
    with output:
        display(Markdown(f'{df['ACCESS Credit Equivalents'].sum():,.1f} ACCESS Credit Equivalents Used'))
        display(Markdown('Usage By Day:'))
        display_timeseries_plot(df)
    return output

def build_tab_widget(titles, children):
    output = ipywidgets.Output()
    tab_widget = ipywidgets.Tab()
    tab_widget.children = children
    tab_widget.titles = titles
    with output:
        display(tab_widget)
    return output

def build_aggregate_and_timeseries_tab(where, group_by):
    df = get_data(
        select=f"{group_by}, SUM(total_ace) AS 'ACCESS Credit Equivalents'",
        where=where,
        group_by=group_by,
    )
    df['Resource'] = all_resources.loc[df['resource_id'], 'label'].to_list()
    df = df[['Resource', 'ACCESS Credit Equivalents']].set_index('Resource')
    print(df)
    aggregate_output = ipywidgets.Output()
    with aggregate_output:
        display_aggregate_plot(df)
    return [aggregate_output, ipywidgets.Output()]

def display_timeseries_plot(
    df,
    title=None,
    color=None,
    labels=None,
    category_orders=None,
    color_discrete_map=None,
    vertical_legend=False,
    yaxis_tickformat=',',
    before_show=None,
    caption='',
):
    plot = px.line(
        df,
        y='ACCESS Credit Equivalents',
        title=title,
        color=color,
        labels=labels,
        category_orders=category_orders,
        color_discrete_map=color_discrete_map,
        # By default, for charts with more than 1000 points, Plotly will switch to rendering with WebGL
        # instead of SVG for better performance. However, there is a browser limit to the number of WebGL
        # contexts that can be rendered on the same web page, and there are enough charts with > 1000 points
        # in this notebook to overrun the limit. Thus, we force SVG, which from manual testing does not seem
        # to have a noticeable hit in performance.
        render_mode='svg',
        height=500,
    )
    plot.update_traces(
        hovertemplate='%{y:,.0f}',
    )
    plot.update_layout(
        yaxis_tickformat=yaxis_tickformat,
        legend_title_text='',
        hovermode='x unified',
        hoverlabel_namelength=-1,
    )
    if vertical_legend:
        plot.update_layout(
            legend_orientation='v',
            legend_xanchor='left',
            legend_x=0,
            legend_yanchor='bottom',
            legend_y=-1.3,
        )
    if before_show is not None:
        before_show(plot)
    #figure_widget = go.FigureWidget(plot)
    #section_number = SECTIONS[CURRENT_SECTION_TITLES[0]]['number']
    #filename = f"ACCESS-RP-Report-{RP.replace(' ', '-')}-{START}-{END}-Fig{section_number}_{CURRENT_FIGURE_NUMBER + 1}"
    plot.show(config={
        #'showAxisRangeEntryBoxes': False,
        #'toImageButtonOptions': {'filename': filename},
    })
    #figure_widget.update_layout(width=1000)
    #caption_widget = get_figure_caption(caption)
    return plot

def display_aggregate_plot(
    df,
    x=None,
    title=None,
    color=None,
    labels=None,
    category_orders=None,
    color_discrete_map=None,
    vertical_legend=False,
    yaxis_tickformat=',',
    before_show=None,
    caption='',
):
    plot = px.bar(
        df,
        x=x,
        y='ACCESS Credit Equivalents',
        title=title,
        color=color,
        labels=labels,
        category_orders=category_orders,
        color_discrete_map=color_discrete_map,
        height=500,
    )
    plot.update_traces(
        hovertemplate='%{y:,.0f}',
    )
    plot.update_layout(
        yaxis_tickformat=yaxis_tickformat,
        legend_title_text='',
        hovermode='x unified',
        hoverlabel_namelength=-1,
    )
    if vertical_legend:
        plot.update_layout(
            legend_orientation='v',
            legend_xanchor='left',
            legend_x=0,
            legend_yanchor='bottom',
            legend_y=-1.3,
        )
    if before_show is not None:
        before_show(plot)
    #figure_widget = go.FigureWidget(plot)
    #section_number = SECTIONS[CURRENT_SECTION_TITLES[0]]['number']
    #filename = f"ACCESS-RP-Report-{RP.replace(' ', '-')}-{START}-{END}-Fig{section_number}_{CURRENT_FIGURE_NUMBER + 1}"
    plot.show(config={
        #'showAxisRangeEntryBoxes': False,
        #'toImageButtonOptions': {'filename': filename},
    })
    #figure_widget.update_layout(width=1000)
    #caption_widget = get_figure_caption(caption)
    return plot

display(build_projects_tab_widget())